[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Pruebas de hipótesis y tamaño del efecto.

**py401 · EDA Práctico** · Pythonista® · Instructor: José Luis Chiquete
**Martes 27 de octubre de 2026 · 6:30 a 8:30 pm (CDMX)** · Módulo III: *Inferencia estadística*

> 📝 **Versión del alumno.** Las celdas marcadas con `# TU CÓDIGO AQUÍ` y las de *Tu respuesta* son para ti.

En esta sesión se aplicarán pruebas *t* de Student y de Welch, Mann-Whitney, pruebas por permutación, la prueba *z* de proporciones, la chi cuadrada de independencia y el ANOVA de una vía con Levene, Welch y Tukey, siempre acompañadas del tamaño del efecto (d de Cohen, V de Cramér y η²), usando `penguins`, `titanic` y `tips`.

## 🎯 Objetivo de la sesión.

**Elegir la prueba adecuada, revisar sus supuestos e informar el tamaño del efecto junto con el valor *p*.**

Al terminar podrás:
1. Explicar H0/H1, el valor *p*, los errores tipo I y II, la potencia y el riesgo de las comparaciones múltiples.
2. Comparar dos grupos con *t* de Welch, Mann-Whitney o permutación, según los datos.
3. Probar proporciones y la independencia de dos variables categóricas.
4. Comparar más de dos grupos con ANOVA, revisar la igualdad de varianzas y localizar diferencias con Tukey.
5. Reportar d de Cohen, V de Cramér y η², e interpretar su magnitud.

## 🗺️ Agenda (120 min).

| Min | Bloque |
|---|---|
| 10 | Repaso |
| 20 | 1. La lógica del contraste |
| 25 | 2. Dos grupos |
| 20 | 3. Proporciones y variables categóricas |
| 25 | 4. Más de dos grupos |
| 20 | 5. Tamaño del efecto y ejercicio de la sesión |

🟢 = ejercicio básico · 🔴 = reto opcional

## 📚 Referencias.

py311 no cubre inferencia, así que la referencia de la API es la documentación oficial: [*SciPy* · `scipy.stats`](https://docs.scipy.org/doc/scipy/reference/stats.html) (`ttest_ind`, `mannwhitneyu`, `permutation_test`, `chi2_contingency`, `f_oneway`, `levene`), [*statsmodels* · pruebas de proporciones](https://www.statsmodels.org/stable/generated/statsmodels.stats.proportion.proportions_ztest.html), [*statsmodels* · `pairwise_tukeyhsd`](https://www.statsmodels.org/stable/generated/statsmodels.stats.multicomp.pairwise_tukeyhsd.html) y [*statsmodels* · `anova_oneway`](https://www.statsmodels.org/stable/generated/statsmodels.stats.oneway.anova_oneway.html).

### Preparación del entorno.

* La siguiente celda importará las bibliotecas y las funciones de *SciPy* y *statsmodels* que se usarán, y definirá `DATA` y la semilla `rng`.

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = next(os.path.join(p, "data") for p in (os.curdir, os.pardir)
            if os.path.isfile(os.path.join(p, "data", "penguins.csv")))
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(401)
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.oneway import anova_oneway
from statsmodels.stats.power import TTestIndPower
from statsmodels.stats.proportion import proportions_ztest

* La siguiente celda leerá `penguins_limpio.csv` (o lo regenerará si no existe), `titanic.csv` y `tips.csv`, y creará `tip_pct`.

In [ ]:
ruta_limpio = os.path.join(DATA, "penguins_limpio.csv")
medidas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
if not os.path.isfile(ruta_limpio):
    (
        pd.read_csv(os.path.join(DATA, "penguins.csv"))
        .assign(sex=lambda d: d["sex"].str.title())
        .dropna(subset=medidas, how="all")
        .to_csv(ruta_limpio, index=False)
    )
pinguinos = pd.read_csv(ruta_limpio)
titanic = pd.read_csv(os.path.join(DATA, "titanic.csv"))
propinas = pd.read_csv(os.path.join(DATA, "tips.csv")).assign(tip_pct=lambda d: 100 * d["tip"] / d["total_bill"])
pinguinos.shape, titanic.shape, propinas.shape

---
## 0. Repaso (10 min).

En la sesión 5, un IC del 95% para la diferencia de masa media entre machos y hembras Gentoo fue de unos 700 a 910 g: el 0 quedó muy lejos. Una prueba de hipótesis responde la pregunta complementaria: **si en la población no hubiera diferencia, ¿qué tan raro sería observar una diferencia como esta?** IC y prueba se complementan; el IC además informa la **magnitud**.

---
## 1. La lógica del contraste (20 min).

| Concepto | Significado |
|---|---|
| H0 (hipótesis nula) | "No hay efecto": diferencia 0, independencia, medias iguales |
| H1 (alternativa) | Hay diferencia (bilateral) o va en una dirección (unilateral, decidida **antes** de ver los datos) |
| Valor *p* | Probabilidad, **si H0 fuera cierta**, de obtener un resultado al menos tan extremo como el observado |
| α (nivel de significancia) | Umbral fijado de antemano (habitualmente 0.05) |
| Error tipo I | Rechazar H0 siendo cierta (probabilidad α) |
| Error tipo II | No rechazar H0 siendo falsa (probabilidad β) |
| Potencia | 1 − β: probabilidad de detectar un efecto que sí existe |

**Advertencia:** el valor *p* **no** es la probabilidad de que H0 sea cierta, no mide el tamaño del efecto y *p* > 0.05 **no** demuestra que no haya diferencia.

**Concepto: cómo se construye una prueba.** Toda prueba de hipótesis sigue los mismos cuatro pasos:

1. Plantear H0 y H1 **antes** de ver los resultados.
2. Elegir un **estadístico de prueba** que mida qué tan lejos están los datos de lo que predice H0. Por ejemplo, para dos medias, $t = \dfrac{\bar{x}_A - \bar{x}_B}{EE(\bar{x}_A - \bar{x}_B)}$: la diferencia observada medida en errores estándar.
3. Obtener su **distribución bajo H0** (con una fórmula, como la *t* de Student, o por simulación, como la permutación).
4. Calcular el valor *p*: $p = P(|T| \geq |t_{obs}| \mid H_0)$. Si $p < \alpha$, se rechaza H0.

Si H0 es cierta, el valor *p* se distribuye **uniforme** entre 0 y 1: hay 5% de probabilidad de que salga menor que 0.05 por puro azar. Eso es exactamente el error tipo I, y es lo que explota el *p-hacking*.

**Ejemplo: comparaciones múltiples y *p-hacking*.** Se simularán 1,000 "estudios" en los que se comparan 20 variables entre dos grupos **idénticos** (H0 es cierta en todas). ¿En qué fracción de estudios aparece al menos un *p* < 0.05?

* La siguiente celda simulará los 1,000 estudios con 20 pruebas *t* cada uno, de forma vectorizada.

In [ ]:
grupo_a = rng.normal(size=(1000, 20, 30))
grupo_b = rng.normal(size=(1000, 20, 30))
valores_p = stats.ttest_ind(grupo_a, grupo_b, axis=2).pvalue
float((valores_p < 0.05).any(axis=1).mean())

Cerca de 64% de los estudios "encuentra algo" (1 − 0.95²⁰ ≈ 0.64) sin que exista ningún efecto. Probar muchas variables, subgrupos o versiones de un análisis y reportar solo lo significativo es *p-hacking*. **Decisiones:** definir las preguntas antes de mirar los resultados, reportar todas las pruebas hechas y, si son muchas, corregir con Holm o Bonferroni (`multipletests`).

* La siguiente celda aplicará la corrección de Holm a las 20 pruebas de cada estudio y calculará en qué fracción de estudios queda algún hallazgo.

In [ ]:
algun_hallazgo_holm = np.array([multipletests(p_estudio, alpha=0.05, method="holm")[0].any() for p_estudio in valores_p])
float(algun_hallazgo_holm.mean())

Con la corrección, la fracción de estudios con algún falso hallazgo vuelve a quedar alrededor de α = 5%. El costo es menos potencia para cada prueba individual: por eso conviene hacer pocas pruebas, bien elegidas.

**Interpretación:** con Holm, solo ≈ 5.6% de los estudios conserva algún falso hallazgo. La corrección de Holm ordena los $m$ valores *p* de menor a mayor y compara el *k*-ésimo con $\alpha / (m - k + 1)$; Bonferroni, más conservadora, compara todos con $\alpha / m$.

### 1.1 Potencia: ¿cuántos datos hacen falta?

La potencia depende del tamaño del efecto, de *n* y de α. `TTestIndPower` de *statsmodels* resuelve cualquiera de ellos dados los demás. El tamaño del efecto se expresa con la **d de Cohen** (diferencia de medias en unidades de desviación estándar).

**Concepto: d de Cohen.** Es la diferencia de medias en unidades de la desviación estándar combinada:

$$d = \frac{\bar{x}_A - \bar{x}_B}{s_p}, \qquad s_p = \sqrt{\frac{(n_A - 1)s_A^2 + (n_B - 1)s_B^2}{n_A + n_B - 2}}$$

Al no tener unidades, permite comparar efectos entre variables distintas y es el insumo de los cálculos de potencia.

* La siguiente celda calculará el *n* por grupo necesario para detectar d = 0.5 con 80% de potencia, y la potencia que se tendría con d = 0.3 y 30 por grupo.

In [ ]:
potencia = TTestIndPower()
(round(potencia.solve_power(effect_size=0.5, alpha=0.05, power=0.8), 1),
 round(float(potencia.power(effect_size=0.3, nobs1=30, alpha=0.05)), 3))

Se necesitan unos 64 casos por grupo para un efecto mediano; con 30 por grupo y un efecto pequeño (d = 0.3), solo se detectaría 1 de cada 5 veces. **Consecuencia:** en un estudio pequeño, "no significativo" casi no informa nada.

**💡 Ideas clave de la sección 1:**
* El valor *p* mide qué tan sorprendentes serían los datos **si H0 fuera cierta**; no es la probabilidad de H0.
* Con muchas pruebas aparecen falsos hallazgos: se planean pocas pruebas y se corrige (Holm).
* La potencia se planea antes: con *n* pequeño, un "no significativo" casi no informa.

### 🟢 Ejercicio 1 · ¿Qué dice un valor *p*?

**🎯 Objetivo:** distinguir la definición del valor *p* de sus lecturas erróneas más comunes. **Qué se espera aprender:** a no confundir *p* con la probabilidad de H0, con la magnitud del efecto ni con una prueba de igualdad.

Una prueba compara la masa de dos colonias y obtiene *p* = 0.03. Indica si cada afirmación es correcta: (a) "hay 3% de probabilidad de que H0 sea cierta"; (b) "si las colonias no difirieran, una diferencia así de grande o mayor ocurriría en torno a 3% de las veces"; (c) "la diferencia es grande"; (d) "con *p* = 0.30 se habría demostrado que las colonias son iguales".

**✍️ Tu respuesta:**

_(escribe aquí)_

---
## 2. Dos grupos (25 min).

| Prueba | Compara | Supuestos | En *SciPy* |
|---|---|---|---|
| *t* de Student | Medias | Independencia, normalidad aproximada (o *n* grande), **varianzas iguales** | `ttest_ind(a, b)` |
| *t* de Welch | Medias | Independencia, normalidad aproximada; **no** exige varianzas iguales | `ttest_ind(a, b, equal_var=False)` |
| Mann-Whitney | Si un grupo tiende a tener valores mayores (rangos) | Independencia; robusta a atípicos y asimetría | `mannwhitneyu(a, b)` |
| Permutación | Cualquier estadístico (diferencia de medias, de medianas…) | Independencia e intercambiabilidad bajo H0 | `permutation_test((a, b), estadistico)` |

**Decisión práctica:** usar **Welch por defecto** para medias; Mann-Whitney o permutación de medianas si hay asimetría fuerte o atípicos con *n* pequeño. La **independencia** no se arregla con ninguna prueba: mediciones repetidas del mismo individuo requieren pruebas pareadas.

**Concepto: Student contra Welch.** Ambas usan $t = (\bar{x}_A - \bar{x}_B) / EE$, pero calculan el error estándar distinto:

$$EE_{\text{Student}} = s_p\sqrt{\frac{1}{n_A} + \frac{1}{n_B}} \qquad EE_{\text{Welch}} = \sqrt{\frac{s_A^2}{n_A} + \frac{s_B^2}{n_B}}$$

Student combina las dos varianzas en una sola $s_p$, ponderada por los tamaños. Si el grupo grande es el poco variable, $s_p$ queda dominada por él y **subestima** el error del grupo pequeño. Welch no combina las varianzas y ajusta los grados de libertad.

**Consecuencia: ¿Student o Welch?** Se simularán 5,000 comparaciones con H0 cierta (medias iguales), pero con un grupo pequeño y muy variable (*n* = 10, σ = 3) y otro grande y poco variable (*n* = 50, σ = 1). Con α = 0.05, la prueba debería equivocarse 5% de las veces.

* La siguiente celda calculará la tasa real de error tipo I de Student y de Welch en ese escenario.

In [ ]:
pequeno_variable = rng.normal(0, 3, size=(5000, 10))
grande_estable = rng.normal(0, 1, size=(5000, 50))
{"Student": float((stats.ttest_ind(pequeno_variable, grande_estable, axis=1).pvalue < 0.05).mean()),
 "Welch": float((stats.ttest_ind(pequeno_variable, grande_estable, axis=1, equal_var=False).pvalue < 0.05).mean())}

Student rechaza H0 en torno a 28% de las veces (¡más de cinco veces α!), mientras que Welch se mantiene en ≈ 5%. Cuando las varianzas son iguales, Welch pierde muy poca potencia; por eso es la opción por defecto.

**Ejemplo:** masa de machos y hembras Adelie. Se revisan los supuestos, se aplican las pruebas y se calcula el tamaño del efecto.

**¿Por qué este orden?** En el flujo del EDA, una comparación de dos grupos se hace en cuatro pasos: (1) describir cada grupo (*n*, centro, dispersión; sesión 4), (2) revisar supuestos, (3) aplicar la prueba y (4) cuantificar la magnitud con su IC. Primero se define la función del tamaño del efecto para usarla al final.

* La siguiente celda definirá `d_cohen`, la diferencia de medias dividida entre la desviación estándar combinada.

In [ ]:
def d_cohen(a, b):
    a, b = np.asarray(a), np.asarray(b)
    varianza_combinada = ((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2)
    return (a.mean() - b.mean()) / np.sqrt(varianza_combinada)

* La siguiente celda separará las masas de machos y hembras Adelie y revisará normalidad (Shapiro-Wilk) e igualdad de varianzas (Levene).

In [ ]:
adelie = pinguinos[pinguinos["species"] == "Adelie"]
machos_adelie = adelie.loc[adelie["sex"] == "Male", "body_mass_g"].to_numpy()
hembras_adelie = adelie.loc[adelie["sex"] == "Female", "body_mass_g"].to_numpy()
{"n": (len(machos_adelie), len(hembras_adelie)),
 "Shapiro machos p": round(float(stats.shapiro(machos_adelie).pvalue), 3),
 "Shapiro hembras p": round(float(stats.shapiro(hembras_adelie).pvalue), 3),
 "Levene p": round(float(stats.levene(machos_adelie, hembras_adelie).pvalue), 3)}

**Interpretación:** con 73 machos y 73 hembras, Shapiro-Wilk no rechaza la normalidad en ningún grupo (*p* = 0.42 y 0.20) y Levene queda en el límite (*p* = 0.051). Como se usará Welch, la igualdad de varianzas no hace falta; la normalidad aproximada y *n* > 30 respaldan la prueba *t*.

**Nota:** las pruebas de normalidad tienen poca potencia con *n* pequeño y rechazan por detalles irrelevantes con *n* grande. Conviene complementarlas con un histograma o un gráfico Q-Q, y recordar que con *n* ≥ 30 por grupo la prueba *t* es robusta a desviaciones moderadas.

**Concepto: tres maneras de obtener la distribución bajo H0.**
* **Welch:** con la fórmula de la *t*, que supone normalidad aproximada de las medias (TCL).
* **Mann-Whitney:** ordena todos los valores juntos y compara los **rangos** de cada grupo con $U = R_A - n_A(n_A + 1)/2$, donde $R_A$ es la suma de rangos del grupo A. Si H0 es cierta, los rangos se reparten al azar. No usa los valores, así que es robusta a atípicos.
* **Permutación:** si H0 es cierta, las etiquetas "macho" y "hembra" son intercambiables. Se revuelven miles de veces, se recalcula la diferencia de medias en cada revuelta y el valor *p* es la fracción de revueltas con una diferencia al menos tan extrema como la observada. No supone ninguna distribución.

* La siguiente celda aplicará Welch, Mann-Whitney y una prueba por permutación de la diferencia de medias (9,999 permutaciones, `random_state=401`).

In [ ]:
welch = stats.ttest_ind(machos_adelie, hembras_adelie, equal_var=False)
mann_whitney = stats.mannwhitneyu(machos_adelie, hembras_adelie)
permutacion = stats.permutation_test((machos_adelie, hembras_adelie),
                                     lambda a, b, axis=-1: np.mean(a, axis=axis) - np.mean(b, axis=axis),
                                     n_resamples=9999, random_state=401)
pd.DataFrame({
    "estadístico": [welch.statistic, mann_whitney.statistic, permutacion.statistic],
    "valor p": [welch.pvalue, mann_whitney.pvalue, permutacion.pvalue],
}, index=["Welch", "Mann-Whitney", "permutación"])

* La siguiente celda calculará la diferencia de medias con su IC del 95% (Welch) y la d de Cohen.

In [ ]:
ic_welch = welch.confidence_interval(confidence_level=0.95)
(round(float(machos_adelie.mean() - hembras_adelie.mean()), 1), np.round([ic_welch.low, ic_welch.high], 1),
 round(float(d_cohen(machos_adelie, hembras_adelie)), 2))

Las tres pruebas coinciden: *p* minúsculo. El valor *p* de la permutación (0.0002) es el mínimo posible con 9,999 permutaciones en una prueba bilateral: ninguna permutación igualó la diferencia observada, así que el valor real es todavía menor. Lo informativo es la magnitud: los machos pesan unos 675 g más (IC 95%: 573 a 776 g), una d de 2.17, un efecto **muy grande**.

| d de Cohen (orientativo) | 0.2 | 0.5 | 0.8 |
|---|---|---|---|
| Magnitud | Pequeño | Mediano | Grande |

**💡 Ideas clave de la sección 2:**
* Welch por defecto; Mann-Whitney o permutación con asimetría o atípicos.
* Los supuestos se revisan con pruebas **y** gráficas; la independencia no se puede "probar", se razona sobre el diseño.
* Se reporta la diferencia con su IC y la d de Cohen, no solo el valor *p*.

### 🟢 Ejercicio 2 · "No significativo" no es "igual".

**🎯 Objetivo:** aplicar Welch con su IC y su tamaño del efecto a un caso con *p* grande. **Qué se espera aprender:** a leer el IC para decir qué diferencias siguen siendo compatibles con los datos cuando no se rechaza H0.

Compara con Welch la masa de los Chinstrap y la de los Adelie. Reporta la diferencia de medias, su IC del 95%, el valor *p* y la d de Cohen. ¿Se puede afirmar que pesan lo mismo?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🟢 Ejercicio 3 · Propina porcentual de fumadores y no fumadores.

**🎯 Objetivo:** elegir pruebas robustas para una variable asimétrica con atípicos. **Qué se espera aprender:** a comparar **medianas** con Mann-Whitney y con una permutación, y a reportar los centros de cada grupo.

`tip_pct` es asimétrica y tiene atípicos (sesión 3). Compara fumadores (`smoker == "Yes"`) y no fumadores con Mann-Whitney y con una prueba por permutación de la **diferencia de medianas** (`random_state=401`). Reporta las medianas de cada grupo.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🔴 Reto 1 · Verifica la potencia por simulación.

**🎯 Objetivo:** comprobar por simulación un cálculo de potencia. **Qué se espera aprender:** qué significa en la práctica "80% de potencia" y cuánto cae al reducir *n*.

`TTestIndPower` dijo que con d = 0.5 hacen falta unos 64 casos por grupo para 80% de potencia. Compruébalo: simula 5,000 pares de muestras de tamaño 64 (una con media 0 y otra con media 0.5, ambas con σ = 1; usa `generador = np.random.default_rng(401)`) y calcula la fracción de pruebas de Welch con *p* < 0.05. Repite con *n* = 20.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

**✍️ Tu respuesta:**

_(escribe aquí)_

---
## 3. Proporciones y variables categóricas (20 min).

* **Prueba *z* de dos proporciones** (`proportions_ztest(conteos, totales)`): ¿difiere la proporción de éxito entre dos grupos?
* **Chi cuadrada de independencia** (`stats.chi2_contingency(tabla)`): ¿están asociadas dos variables categóricas? Compara la tabla observada con la **esperada** si fueran independientes.
* **Supuesto clave:** frecuencias **esperadas** suficientemente grandes (en general ≥ 5 en cada celda). Si no se cumple, se agrupan categorías o se usa la prueba exacta de Fisher (`stats.fisher_exact`, para tablas 2×2).

**Concepto: prueba *z* de dos proporciones.** Bajo H0 ($p_A = p_B$) se estima una proporción común $\hat{p}$ con los dos grupos juntos y

$$z = \frac{\hat{p}_A - \hat{p}_B}{\sqrt{\hat{p}(1-\hat{p})\left(\frac{1}{n_A} + \frac{1}{n_B}\right)}}$$

que, por el TCL, se compara con la normal estándar.

* La siguiente celda comparará con la prueba *z* la tasa de supervivencia de mujeres y hombres en el Titanic.

In [ ]:
tabla_sexo = pd.crosstab(titanic["sex"], titanic["survived"])
sobrevivientes = tabla_sexo[1].to_numpy()
totales = tabla_sexo.sum(axis=1).to_numpy()
estadistico_z, valor_p_z = proportions_ztest(sobrevivientes, totales)
(sobrevivientes / totales).round(3), round(float(estadistico_z), 2), float(valor_p_z)

La diferencia es enorme (74.2% contra 18.9%) y su valor *p* es prácticamente 0. Para reportar la magnitud se usa la **diferencia de proporciones** con su IC (sesión 5) o el **riesgo relativo**: las mujeres sobrevivieron casi 4 veces más que los hombres.

**Ejemplo:** ¿están asociadas la clase y la supervivencia? Se revisa la tabla esperada antes de interpretar.

**Concepto: chi cuadrada.** Si las variables fueran independientes, la frecuencia esperada de cada celda sería $E_{ij} = \dfrac{(\text{total de la fila } i)(\text{total de la columna } j)}{n}$. El estadístico suma las discrepancias relativas:

$$\chi^2 = \sum_{i,j} \frac{(O_{ij} - E_{ij})^2}{E_{ij}}, \qquad g.l. = (f - 1)(c - 1)$$

y su tamaño del efecto es la **V de Cramér**, $V = \sqrt{\dfrac{\chi^2}{n\,(\min(f, c) - 1)}}$, que va de 0 a 1.

* La siguiente celda aplicará la chi cuadrada de independencia a `class` contra `survived` y mostrará la tabla esperada.

In [ ]:
tabla_clase = pd.crosstab(titanic["class"], titanic["survived"])
chi2 = stats.chi2_contingency(tabla_clase)
esperada = pd.DataFrame(chi2.expected_freq, index=tabla_clase.index, columns=tabla_clase.columns).round(1)
round(float(chi2.statistic), 1), int(chi2.dof), float(chi2.pvalue), esperada

Todas las frecuencias esperadas superan 5, así que la aproximación es válida. La chi cuadrada dice que hay asociación, pero no cuánta: el tamaño del efecto es la **V de Cramér** (0 = independencia, 1 = asociación perfecta), que se calcula con `stats.contingency.association(tabla, method="cramer")`.

| V de Cramér con 1 grado de libertad mínimo (orientativo) | 0.1 | 0.3 | 0.5 |
|---|---|---|---|
| Magnitud | Pequeña | Mediana | Grande |

**💡 Ideas clave de la sección 3:**
* Prueba *z* para dos proporciones; chi cuadrada para dos categóricas, revisando que las frecuencias esperadas sean ≥ 5.
* La magnitud se reporta con la diferencia de proporciones, el riesgo relativo o la V de Cramér.

### 🟢 Ejercicio 4 · Mujeres y hombres en tercera clase.

**🎯 Objetivo:** aplicar la prueba *z* y la V de Cramér dentro de un subgrupo. **Qué se espera aprender:** a revisar si una asociación global se mantiene al controlar por otra variable.

¿La ventaja de las mujeres se mantiene en tercera clase? Con los pasajeros de `class == "Third"`, compara la supervivencia de mujeres y hombres con la prueba *z* y calcula la V de Cramér de la tabla 2×2.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

---
## 4. Más de dos grupos (25 min).

Comparar tres grupos con tres pruebas *t* infla el error tipo I (sección 1). El procedimiento es:

1. **ANOVA de una vía** (`stats.f_oneway`): H0 = todas las medias son iguales. Supone independencia, normalidad aproximada y varianzas iguales.
2. **Levene** (`stats.levene`): ¿son iguales las varianzas? Si no, se usa el **ANOVA de Welch** (`anova_oneway(..., use_var="unequal")` de *statsmodels*).
3. Si se rechaza H0, **post hoc de Tukey** (`pairwise_tukeyhsd`) para saber **qué** pares difieren, controlando el error de las comparaciones múltiples.
4. Tamaño del efecto: **η²** = suma de cuadrados entre grupos / suma de cuadrados total (proporción de la variabilidad explicada por el grupo).

**Concepto: ANOVA y η².** La variabilidad total se descompone en una parte **entre** grupos y otra **dentro** de los grupos:

$$SC_{\text{total}} = \sum_{i}(x_i - \bar{x})^2 = \underbrace{\sum_{g} n_g(\bar{x}_g - \bar{x})^2}_{SC_{\text{entre}}} + \underbrace{\sum_{g}\sum_{i \in g}(x_i - \bar{x}_g)^2}_{SC_{\text{dentro}}}$$

El estadístico $F = \dfrac{SC_{\text{entre}} / (k - 1)}{SC_{\text{dentro}} / (n - k)}$ compara ambas partes; si los grupos no difieren, $F \approx 1$. El tamaño del efecto es $\eta^2 = SC_{\text{entre}} / SC_{\text{total}}$, la fracción de la variabilidad que se explica por el grupo.

* La siguiente celda definirá `eta_cuadrada` a partir de los valores y las etiquetas de grupo.

In [ ]:
def eta_cuadrada(valores, grupos):
    datos = pd.DataFrame({"valor": valores, "grupo": grupos}).dropna()
    media_global = datos["valor"].mean()
    por_grupo = datos.groupby("grupo")["valor"].agg(["count", "mean"])
    suma_entre = (por_grupo["count"] * (por_grupo["mean"] - media_global) ** 2).sum()
    suma_total = ((datos["valor"] - media_global) ** 2).sum()
    return suma_entre / suma_total

**Ejemplo:** se usará `tip_pct` por día de la semana en `tips`, un caso donde **no** se espera una diferencia grande.

* La siguiente celda aplicará Levene, el ANOVA clásico, el ANOVA de Welch y Kruskal-Wallis (alternativa por rangos) a `tip_pct` por día, y calculará η².

In [ ]:
grupos_dia = [grupo["tip_pct"].to_numpy() for _, grupo in propinas.groupby("day")]
welch_dia = anova_oneway(propinas["tip_pct"], propinas["day"], use_var="unequal")
pd.DataFrame({
    "valor p": [stats.levene(*grupos_dia).pvalue, stats.f_oneway(*grupos_dia).pvalue,
                welch_dia.pvalue, stats.kruskal(*grupos_dia).pvalue],
}, index=["Levene", "ANOVA", "ANOVA de Welch", "Kruskal-Wallis"]).round(3), round(float(eta_cuadrada(propinas["tip_pct"], propinas["day"])), 3)

Ninguna prueba encuentra diferencias y η² es casi 0: el día explica apenas alrededor de 1% de la variabilidad de la propina porcentual. Como el ANOVA no rechaza, **no** se hace Tukey: buscar pares significativos después de un ANOVA no significativo es otra forma de *p-hacking*.

| η² (orientativo) | 0.01 | 0.06 | 0.14 |
|---|---|---|---|
| Magnitud | Pequeño | Mediano | Grande |

**Consecuencia: significativo no es lo mismo que importante.** Con muestras enormes, cualquier diferencia mínima resulta significativa. Se simularán dos grupos de 100,000 personas cuyas medias difieren 0.45 puntos en una escala con σ = 15.

* La siguiente celda calculará el valor *p* de Welch y la d de Cohen de esa diferencia mínima.

In [ ]:
enorme_a = rng.normal(100, 15, size=100_000)
enorme_b = rng.normal(100.45, 15, size=100_000)
float(stats.ttest_ind(enorme_b, enorme_a, equal_var=False).pvalue), round(float(d_cohen(enorme_b, enorme_a)), 3)

El valor *p* es diminuto, pero la d es ≈ 0.03: un efecto **despreciable**. Reportar solo "*p* < 0.001" llevaría a creer que es un hallazgo importante. Por eso el tamaño del efecto y su IC son obligatorios en el informe.

**💡 Ideas clave de la sección 4:**
* ANOVA (Welch si Levene rechaza) → Tukey solo si el ANOVA rechaza → η².
* Kruskal-Wallis es la alternativa por rangos.
* Con *n* enorme todo es significativo: el tamaño del efecto decide si importa.

### 🔴 Reto 2 · Doce pruebas a la vez.

**🎯 Objetivo:** aplicar correcciones por comparaciones múltiples a un caso real. **Qué se espera aprender:** cuándo las correcciones cambian las conclusiones y cuándo no.

Para cada especie y cada una de las cuatro medidas, compara machos y hembras con Welch (12 pruebas). Aplica las correcciones de Bonferroni y de Holm con `multipletests`. ¿Cuántas pruebas siguen siendo significativas? Agrega la d de Cohen de cada comparación: ¿cuál es la diferencia entre sexos más pequeña?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

**✍️ Tu respuesta:**

_(escribe aquí)_

---
## 5. Tamaño del efecto y ejercicio de la sesión (20 min).

| Situación | Prueba | Tamaño del efecto |
|---|---|---|
| Dos medias | Welch | d de Cohen (y la diferencia con su IC, en unidades originales) |
| Dos distribuciones asimétricas | Mann-Whitney o permutación de medianas | Diferencia de medianas con IC *bootstrap* |
| Dos proporciones | *z* de proporciones | Diferencia de proporciones o riesgo relativo |
| Dos categóricas | Chi cuadrada | V de Cramér |
| Tres o más medias | ANOVA (Welch si las varianzas difieren) + Tukey | η² |

**Plantilla para redactar:** "En estos datos, [grupo A] tuvo [medida] [mayor/menor] que [grupo B] ([valor A] contra [valor B]; diferencia [d], IC 95% [a, b]; [prueba], *p* [valor]; [tamaño del efecto] = [valor], un efecto [magnitud])." Sin verbos causales ("provoca", "hace que") cuando los datos son observacionales.

**¿Por qué esta sección?** El producto final de una prueba no es el valor *p*, sino una **frase** que una persona sin formación estadística pueda leer y que no exagere. La tabla resume qué tamaño del efecto acompaña a cada prueba, y la plantilla es la que se usará en el informe del proyecto (sesiones 7 y 8).

**💡 Ideas clave de la sección 5:** cada prueba tiene su tamaño del efecto; se reporta magnitud, IC, prueba y valor *p*, en lenguaje descriptivo.

### 🟢 Ejercicio 5a · Masa por sexo en Adelie.

**🎯 Objetivo:** redactar un resultado con la plantilla. **Qué se espera aprender:** a comunicar magnitud, incertidumbre y prueba en una sola frase, sin lenguaje causal.

Con los resultados de la sección 2, redacta la conclusión de la comparación de masa entre machos y hembras Adelie en una o dos frases, con la plantilla.

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🟢 Ejercicio 5b · Aleta por especie.

**🎯 Objetivo:** recorrer el flujo completo para más de dos grupos (Levene → ANOVA → Tukey → η²). **Qué se espera aprender:** a decidir cada paso según el anterior y a redactar la conclusión con el tamaño del efecto.

Compara `flipper_length_mm` entre las tres especies: revisa Levene, aplica el ANOVA (y el de Welch), calcula η² y, si corresponde, Tukey. Redacta la conclusión.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🟢 Ejercicio 5c · Clase y supervivencia.

**🎯 Objetivo:** cuantificar una asociación entre categóricas. **Qué se espera aprender:** a acompañar la chi cuadrada con la V de Cramér y las tasas por grupo.

Con la chi cuadrada de la sección 3, calcula la V de Cramér de `class` contra `survived` y redacta la conclusión sin lenguaje causal injustificado.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🔴 Reto 3 · Diagnóstico gráfico de supuestos.

**🎯 Objetivo:** revisar los supuestos del ANOVA con gráficas. **Qué se espera aprender:** a juzgar si una desviación de los supuestos puede cambiar la conclusión.

Para el ejercicio 5b, dibuja un gráfico Q-Q de `flipper_length_mm` de cada especie (`stats.probplot(..., plot=ax)`) y un diagrama de caja por especie. ¿Las conclusiones del ANOVA dependen de algún supuesto dudoso?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

**✍️ Tu respuesta:**

_(escribe aquí)_

---
## ✅ Cierre.

**Hoy vimos:**
* La lógica del contraste, qué **no** significa un valor *p*, la potencia y por qué las comparaciones múltiples producen falsos hallazgos.
* Dos grupos: Welch por defecto (Student falla con varianzas y tamaños distintos), Mann-Whitney y permutación.
* Proporciones (*z*) y categóricas (chi cuadrada, revisando las frecuencias esperadas).
* Más de dos grupos: Levene, ANOVA clásico y de Welch, Tukey solo si el ANOVA rechaza.
* Tamaño del efecto: d de Cohen, V de Cramér y η²; con *n* enorme, significativo no implica importante.

**Práctica (sin calificación, en Classroom):** completar los tres incisos del ejercicio 5 y aplicar al menos una prueba con su tamaño del efecto al *dataset* del proyecto.

**Próxima sesión (mié 28 oct):** EDA completo paso a paso con `mpg` y arranque del proyecto (`07_eda_completo.ipynb`).

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>